The visualisations produced in this notebook will be copied into `evaluation.md` and discussed there, with an overview in `README.md`

# Visualising evaluation results

Charts for one evaluation run, built from the CSVs written by `run_evaluation`:

- `results/run_<label>_refusals.csv`: one row per question, with its hand-labelled `refusal` and its outcome
- `results/run_<label>_retrieval.csv`: one row per answerable question whose evidence is findable, with recall@k and MRR

Set `LABEL` below to the run you want to look at.

In [ ]:
import ast

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch, Rectangle

from rag_pipeline.run_evaluation import RESULTS_DIR

LABEL = "baseline"

# Colours (light chart surface; text in ink colours, never the series colour)
SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
NEUTRAL_FILL = "#f0efec"
BLUE = "#2a78d6"
ORANGE = "#eb6834"  # second categorical slot

# Outcomes use the status palette, always with a text label or legend entry.
# The two correct outcomes share "good" and are told apart by texture.
OUTCOMES = {
    "answered": {"label": "Answered", "color": "#0ca30c", "hatch": None},
    "correct_refusal": {"label": "Correct refusal", "color": "#0ca30c", "hatch": "///"},
    "false_refusal": {"label": "False refusal", "color": "#ec835a", "hatch": None},
    "missed_refusal": {"label": "Missed refusal", "color": "#d03b3b", "hatch": None},
}

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": TEXT_SECONDARY,
        "axes.titlecolor": TEXT,
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "xtick.color": TEXT_SECONDARY,
        "ytick.color": TEXT_SECONDARY,
        "text.color": TEXT,
        "font.size": 10,
        "hatch.color": SURFACE,
        "hatch.linewidth": 1.5,
    }
)


def _style_axes(ax, grid_axis=None):
    """Recessive axes: no top/right spines, optional hairline grid behind the marks."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)


def _subtitle(ax, text):
    ax.text(
        0,
        1.02,
        text,
        transform=ax.transAxes,
        color=TEXT_SECONDARY,
        fontsize=10,
        va="bottom",
    )

## Load the results

The retrieval CSV only covers answerable questions, so joining it onto the refusal CSV gives each answerable question both its retrieval scores and whether the model refused it.

In [ ]:
refusals = pd.read_csv(RESULTS_DIR / f"run_{LABEL}_refusals.csv")
retrieval = pd.read_csv(RESULTS_DIR / f"run_{LABEL}_retrieval.csv")
retrieval["evidence_ranks"] = retrieval["evidence_ranks"].apply(ast.literal_eval)

# Recall at the largest k is recall over everything the generator saw.
K_MAX = max(int(c.split("@")[1]) for c in retrieval.columns if c.startswith("recall@"))

answerable = retrieval.merge(
    refusals[["id", "refusal", "outcome"]], on="id", how="left"
)
answerable["row_label"] = answerable["id"] + " · " + answerable["category"]

display(refusals)
display(answerable.drop(columns="row_label"))

## 1. Refusal outcomes

Each question lands in one of four cells. The diagonal (top left, bottom right) is correct behaviour. **False refusals** are answerable questions the model refused, and **missed refusals** are unanswerable questions it answered anyway, which makes them the likeliest hallucinations.

In [ ]:
def _rate(n, d):
    return f"{n}/{d} ({n / d:.0%})" if d else "n/a"


def plot_refusal_matrix(refusals):
    counts = refusals["outcome"].value_counts()
    n = lambda o: int(counts.get(o, 0))  # noqa: E731
    n_unans = n("correct_refusal") + n("missed_refusal")
    n_ans = n("answered") + n("false_refusal")

    # rows: answerable yes / no; columns: answered / refused
    grid = [["answered", "false_refusal"], ["missed_refusal", "correct_refusal"]]

    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    for r, row in enumerate(grid):
        for c, outcome in enumerate(row):
            style = OUTCOMES[outcome]
            count = n(outcome)
            y = 1 - r
            ax.add_patch(
                Rectangle(
                    (c + 0.03, y + 0.03),
                    0.94,
                    0.94,
                    facecolor=style["color"] if count else NEUTRAL_FILL,
                    alpha=0.18 if count else 1,
                    edgecolor="none",
                )
            )
            ax.add_patch(
                Rectangle(
                    (c + 0.03, y + 0.03),
                    0.94,
                    0.94,
                    fill=False,
                    edgecolor=style["color"] if count else AXIS,
                    linewidth=2,
                )
            )
            ax.text(
                c + 0.5,
                y + 0.56,
                str(count),
                ha="center",
                va="center",
                fontsize=28,
                fontweight="bold",
            )
            ax.text(
                c + 0.5,
                y + 0.28,
                style["label"],
                ha="center",
                va="center",
                color=TEXT_SECONDARY,
                fontsize=11,
            )

    ax.set_xlim(0, 2)
    ax.set_ylim(0, 2)
    ax.set_xticks([0.5, 1.5], ["Answered", "Refused"])
    ax.set_yticks([1.5, 0.5], ["Answerable", "Unanswerable"])
    ax.xaxis.tick_top()
    ax.tick_params(length=0, labelsize=11)
    ax.spines[:].set_visible(False)
    ax.set_title(f"Refusal outcomes · {len(refusals)} questions", pad=44)
    ax.text(
        0,
        1.13,
        f"Refusal rate {_rate(n('correct_refusal'), n_unans)}   ·   "
        f"Missed refusal rate {_rate(n('missed_refusal'), n_unans)}   ·   "
        f"False refusal rate {_rate(n('false_refusal'), n_ans)}",
        transform=ax.transAxes,
        color=TEXT_SECONDARY,
        fontsize=9.5,
    )
    ax.set_aspect("equal")
    fig.tight_layout()
    return fig


plot_refusal_matrix(refusals);

## 2. Outcomes by category

Which topics drive the refusals? With only a question or two per category this is anecdotal rather than statistical, but it points to where to look first, and it shows which categories need more questions in the dataset.

In [ ]:
def plot_outcomes_by_category(refusals):
    table = (
        refusals.groupby(["category", "outcome"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=list(OUTCOMES), fill_value=0)
    )
    table = table.loc[table.sum(axis=1).sort_values().index]  # largest at top

    fig, ax = plt.subplots(figsize=(8, 0.42 * len(table) + 1.6))
    left = pd.Series(0, index=table.index)
    for outcome, style in OUTCOMES.items():
        values = table[outcome]
        ax.barh(
            table.index,
            values,
            left=left,
            height=0.62,
            color=style["color"],
            hatch=style["hatch"],
            edgecolor=SURFACE,
            linewidth=2,
            label=style["label"],
        )
        left += values

    ax.set_xlim(0, table.sum(axis=1).max() + 0.2)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_xlabel("Questions")
    _style_axes(ax, grid_axis="x")
    ax.spines["left"].set_visible(False)
    ax.set_title("Outcomes by category", pad=30)
    ax.legend(
        handles=[
            Patch(
                facecolor=s["color"],
                hatch=s["hatch"],
                edgecolor=SURFACE,
                label=s["label"],
            )
            for s in OUTCOMES.values()
        ],
        loc="lower left",
        bbox_to_anchor=(0, 1.0),
        ncol=4,
        frameon=False,
        handlelength=1.4,
        columnspacing=1.2,
        fontsize=9.5,
    )
    fig.tight_layout()
    return fig


plot_outcomes_by_category(refusals);

## 3. Was a false refusal a retrieval failure or a generation failure?

Each answerable question's recall at the generator's k, coloured by whether the model answered it.

- **Refused with recall 0**: the evidence never reached the model, so refusing was arguably the *right* response to what it was given. Fix retrieval.
- **Refused with recall 1**: all the evidence was in the context and the model still refused. Fix the prompt or the generator.
- **Answered with low recall**: the answer may be incomplete. Check it against the reference answer.

In [ ]:
def plot_retrieval_vs_refusal(answerable, k):
    df = answerable.sort_values(
        [f"recall@{k}", "id"], ascending=[True, False]
    ).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 0.42 * len(df) + 1.8))
    ax.axvspan(-0.06, 0.06, color=NEUTRAL_FILL, zorder=0)
    ax.axvspan(0.94, 1.06, color=NEUTRAL_FILL, zorder=0)
    ax.text(
        0,
        len(df) - 0.35,
        "retrieval\nmiss",
        ha="center",
        va="bottom",
        color=TEXT_SECONDARY,
        fontsize=8.5,
    )
    ax.text(
        1,
        len(df) - 0.35,
        "all evidence\nretrieved",
        ha="center",
        va="bottom",
        color=TEXT_SECONDARY,
        fontsize=8.5,
    )

    for y, row in df.iterrows():
        style = OUTCOMES[row["outcome"]]
        refused = row["outcome"] == "false_refusal"
        x = row[f"recall@{k}"]
        ax.plot([0, x], [y, y], color=GRID, linewidth=2, zorder=1)
        ax.scatter(
            x,
            y,
            s=90,
            zorder=3,
            linewidths=2,
            facecolor=SURFACE if refused else style["color"],
            edgecolor=style["color"],
        )
        ax.annotate(
            "refused" if refused else "answered",
            (x, y),
            xytext=(10, 0),
            textcoords="offset points",
            va="center",
            fontsize=9,
            color=TEXT if refused else TEXT_SECONDARY,
            fontweight="bold" if refused else "normal",
        )

    ax.set_yticks(range(len(df)), df["row_label"])
    ax.set_xlim(-0.08, 1.22)
    ax.set_ylim(-0.6, len(df) + 0.6)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1], ["0", "0.25", "0.5", "0.75", "1"])
    ax.set_xlabel(
        f"Recall@{k} (share of evidence items in the {k} chunks the model saw)"
    )
    _style_axes(ax, grid_axis="x")
    ax.spines["left"].set_visible(False)
    ax.set_title("Retrieval recall vs. whether the model answered", pad=24)
    _subtitle(ax, "Filled = answered · hollow = refused (false refusal)")
    fig.tight_layout()
    return fig


plot_retrieval_vs_refusal(answerable, K_MAX);

## 4. Retrieval summary: recall@k and MRR

The headline retrieval numbers. Recall@k is averaged two ways:

- **Mean per question** (what `run_evaluation` prints): each question counts equally, however many evidence items it has.
- **Pooled over evidence items**: each evidence item counts equally, so a question with 7 items (like q001) weighs 7 times as much as one with 1.

If the two lines differ a lot, a few evidence-heavy questions are pulling the result one way. The gap between recall@1 and recall@k shows how much a re-ranker could gain: evidence that is retrieved, but not near the top.

In [ ]:
def plot_recall_at_k(retrieval):
    ks = sorted(
        int(c.split("@")[1]) for c in retrieval.columns if c.startswith("recall@")
    )
    ranks = [r for rs in retrieval["evidence_ranks"] for r in rs]
    per_question = [retrieval[f"recall@{k}"].mean() for k in ks]
    pooled = [sum(r is not None and r <= k for r in ranks) / len(ranks) for k in ks]
    mrr = retrieval["reciprocal_rank"].mean()

    fig, ax = plt.subplots(figsize=(7, 4.4))
    series = [
        ("Mean per question", per_question, BLUE, "o"),
        ("Pooled over evidence items", pooled, ORANGE, "s"),
    ]
    other = {0: pooled, 1: per_question}
    for i, (label, values, color, marker) in enumerate(series):
        ax.plot(
            ks,
            values,
            color=color,
            linewidth=2,
            marker=marker,
            markersize=8,
            markeredgecolor=SURFACE,
            markeredgewidth=2,
            label=label,
        )
        # Label above the point if this series is the higher one at k, else below.
        for k, v, o in zip(ks, values, other[i]):
            ax.annotate(
                f"{v:.2f}",
                (k, v),
                xytext=(0, 9) if v >= o else (0, -16),
                textcoords="offset points",
                ha="center",
                fontsize=9,
                color=TEXT_SECONDARY,
            )

    ax.set_xticks(ks, [f"k = {k}" for k in ks])
    ax.set_xlim(ks[0] - 0.4, ks[-1] + 0.4)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Recall@k")
    _style_axes(ax, grid_axis="y")
    ax.set_title("Recall@k", pad=46)
    _subtitle(
        ax,
        f"MRR@{ks[-1]} {mrr:.2f}   ·   {len(retrieval)} questions, "
        f"{len(ranks)} evidence items",
    )
    ax.legend(
        loc="lower left", bbox_to_anchor=(0, 1.07), ncol=2, frameon=False, fontsize=9.5
    )
    fig.tight_layout()
    return fig


plot_recall_at_k(retrieval);

## 5. Retrieval metrics per question

Every retrieval metric for every answerable question, shaded from white (0) to dark blue (1). Read across a row to see how a question's recall builds up as k grows. A row that's pale throughout is a retrieval miss. A row that only darkens at the right-hand recall column has its evidence retrieved, but low down. RR (reciprocal rank) is 1 / the rank of the first evidence item retrieved.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

BLUE_RAMP = LinearSegmentedColormap.from_list(
    "blue_ramp", [NEUTRAL_FILL, "#cde2fb", "#6da7ec", "#256abf", "#0d366b"]
)


def plot_retrieval_heatmap(retrieval):
    ks = sorted(
        int(c.split("@")[1]) for c in retrieval.columns if c.startswith("recall@")
    )
    cols = [f"recall@{k}" for k in ks] + ["reciprocal_rank"]
    df = retrieval.sort_values("id").reset_index(drop=True)
    values = df[cols].astype(float).to_numpy()

    fig, ax = plt.subplots(figsize=(7, 0.42 * len(df) + 1.6))
    ax.imshow(values, cmap=BLUE_RAMP, vmin=0, vmax=1, aspect="auto")
    # 2px surface gaps between cells
    ax.set_xticks([x - 0.5 for x in range(1, len(cols))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(df))], minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="both", length=0)

    for y in range(len(df)):
        for x in range(len(cols)):
            v = values[y, x]
            ax.text(
                x,
                y,
                f"{v:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                color=SURFACE if v >= 0.6 else TEXT,
            )

    ax.set_xticks(range(len(cols)), [f"Recall@{k}" for k in ks] + ["RR"])
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(df)), df["id"] + " · " + df["category"])
    ax.spines[:].set_visible(False)
    ax.set_title("Retrieval metrics per question", pad=34)
    fig.tight_layout()
    return fig


plot_retrieval_heatmap(retrieval);

## 6. RAGAS: faithfulness and answer relevancy

One row per answerable question. The dashed line in each panel is the mean over the questions that were scored.

- **Faithfulness**: the share of the answer's claims that the retrieved context supports. A low score means the model added something the context doesn't say.
- **Answer relevancy**: how well the answer addresses the question. RAGAS sets it to **0** if the judge thinks the answer is *non-committal* (hedged or evasive), so a 0 can mean "hedged" rather than "off-topic". Read the answer before trusting a 0.

Only answerable questions that the model answered are sent to RAGAS. False refusals show as "refused" here: they are counted in the false refusal rate instead. Because of this, the RAGAS means can go up just because the model refused more of the hard questions. Always read them next to the false refusal rate.

Neither metric checks **completeness**: an answer that leaves out an important warning can still score 1 on both.

In [ ]:
RAGAS_METRICS = {"faithfulness": "Faithfulness", "answer_relevancy": "Answer relevancy"}

ragas_path = RESULTS_DIR / f"run_{LABEL}_ragas.csv"
ragas = pd.read_csv(ragas_path) if ragas_path.exists() else None


def plot_generation_scores(refusals, ragas):
    df = (
        refusals.loc[refusals["answerable"].astype(bool), ["id", "category", "outcome"]]
        .merge(ragas[["id", *RAGAS_METRICS]], on="id", how="left")
        .sort_values("id", ascending=False)
        .reset_index(drop=True)
    )

    fig, axes = plt.subplots(
        1, len(RAGAS_METRICS), figsize=(8, 0.42 * len(df) + 2), sharey=True
    )
    for ax, (col, title) in zip(axes, RAGAS_METRICS.items()):
        scored = df[col].dropna()
        for y, row in df.iterrows():
            v = row[col]
            if pd.isna(v):
                refused = row["outcome"] == "false_refusal"
                ax.text(
                    0.5,
                    y,
                    "refused" if refused else "not scored",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color=TEXT_SECONDARY,
                    style="italic",
                )
                continue
            ax.plot([0, v], [y, y], color=GRID, linewidth=2, zorder=1)
            ax.scatter(
                v, y, s=80, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=3
            )
            ax.annotate(
                f"{v:.2f}",
                (v, y),
                xytext=(8, 0),
                textcoords="offset points",
                va="center",
                fontsize=9,
                color=TEXT_SECONDARY,
            )
        if len(scored):
            ax.axvline(
                scored.mean(),
                color=TEXT_SECONDARY,
                linewidth=1,
                linestyle=(0, (3, 3)),
                zorder=2,
            )
        ax.set_title(title, fontsize=11, pad=22)
        ax.text(
            0,
            1.01,
            f"mean {scored.mean():.2f} · n = {len(scored)}"
            if len(scored)
            else "no scores",
            transform=ax.transAxes,
            fontsize=9,
            color=TEXT_SECONDARY,
            va="bottom",
        )
        ax.set_xlim(-0.05, 1.3)
        ax.set_xticks([0, 0.5, 1], ["0", "0.5", "1"])
        ax.set_ylim(-0.6, len(df) - 0.4)
        _style_axes(ax, grid_axis="x")
        ax.spines["left"].set_visible(False)

    axes[0].set_yticks(range(len(df)), df["id"] + " · " + df["category"])
    fig.suptitle(
        "RAGAS scores per answerable question",
        x=0.01,
        ha="left",
        fontsize=13,
        fontweight="bold",
    )
    fig.tight_layout()
    return fig


if ragas is None:
    print(f"No RAGAS results at {ragas_path} (was the run scored with --skip-ragas?)")
else:
    plot_generation_scores(refusals, ragas)